# Transfer Learning Models

### Import Libraries

In [1]:
# Import Libraries

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from tensorflow.keras.applications import (
    ResNet50,
    MobileNet,
    EfficientNetB0
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    GlobalAveragePooling2D,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Recreate Generators in Transfer learning models Notebook

In [2]:
# code

# Dataset Paths

train_dir = "../data/train"
val_dir = "../data/val"
test_dir = "../data/test"

# Image Parameters

IMAGE_SIZE = (224, 224)

BATCH_SIZE = 32

EPOCHS = 20

# Data Augmentation

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())

print(class_names)


Found 1862 images belonging to 2 classes.
Found 399 images belonging to 2 classes.
Found 401 images belonging to 2 classes.
['bird', 'drone']


### Define Common Parameters

In [3]:
# Common Parameters

IMAGE_SIZE = (224, 224)

NUM_CLASSES = 2

EPOCHS = 15

### Helper Function (Training + Evaluation) -> (Reusable)

In [4]:
# code -> core reusable function

def train_and_evaluate_model(
    base_model,
    model_name
):

    print("\nTraining:", model_name)

    base_model.trainable = False

    x = base_model.output

    x = GlobalAveragePooling2D()(x)

    x = Dense(128, activation="relu")(x)

    x = Dropout(0.5)(x)

    predictions = Dense(
        2,
        activation="softmax"
    )(x)

    model = Model(
        inputs=base_model.input,
        outputs=predictions
    )

    model.compile(

        optimizer=Adam(
            learning_rate=0.0001
        ),

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )

    callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),

        ModelCheckpoint(
            f"../models/{model_name}.keras",
            save_best_only=True
        )

    ]

    start_time = time.time()

    history = model.fit(

        train_generator,

        validation_data=val_generator,

        epochs=20,

        callbacks=callbacks

    )

    end_time = time.time()

    training_time = (
        end_time - start_time
    ) / 60

    print("Training time:", training_time)

    y_pred = model.predict(
        test_generator
    )

    y_pred_classes = np.argmax(
        y_pred,
        axis=1
    )

    report = classification_report(

        test_generator.classes,

        y_pred_classes,

        target_names=class_names,

        output_dict=True

    )

    accuracy = report["accuracy"]

    precision = report[
        "weighted avg"
    ]["precision"]

    recall = report[
        "weighted avg"
    ]["recall"]

    f1 = report[
        "weighted avg"
    ]["f1-score"]

    model_path = (
        f"../models/{model_name}.keras"
    )

    size = os.path.getsize(
        model_path
    ) / (1024 * 1024)

    print("Accuracy:", accuracy)

    print("Precision:", precision)

    print("Recall:", recall)

    print("F1:", f1)

    print("Model Size:", size)

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "training_time": training_time,

        "size": size

    }

### Train ResNet50 Model

In [5]:
# code

resnet_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(224, 224, 3)

)

resnet_results = train_and_evaluate_model(

    resnet_model,

    "resnet50_model"

)


Training: resnet50_model
Epoch 1/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.5161 - loss: 0.8385 - val_accuracy: 0.6642 - val_loss: 0.6308
Epoch 2/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - accuracy: 0.5725 - loss: 0.7036 - val_accuracy: 0.6491 - val_loss: 0.6187
Epoch 3/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 179s 3s/step - accuracy: 0.5940 - loss: 0.6761 - val_accuracy: 0.6642 - val_loss: 0.6200
Epoch 4/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 166s 3s/step - accuracy: 0.6224 - loss: 0.6508 - val_accuracy: 0.6742 - val_loss: 0.6152
Epoch 5/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 174s 3s/step - accuracy: 0.6353 - loss: 0.6367 - val_accuracy: 0.6566 - val_loss: 0.6075
Epoch 6/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 173s 2s/step - accuracy: 0.6455 - loss: 0.6356 - val_accuracy: 0.6642 - val_loss: 0.6056
Epoch 7/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 574s 10s/step - accuracy: 0.6429 - loss: 0.6310 - val_accuracy: 0.6767 - val_loss: 0.6104
Epoch 8/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 153s 3s/step - accuracy: 0.6472 - loss: 0.6276

### Train MobileNet Model

In [6]:
# code

mobilenet_model = MobileNet(

    weights="imagenet",

    include_top=False,

    input_shape=(224, 224, 3)

)

mobilenet_results = train_and_evaluate_model(

    mobilenet_model,

    "mobilenet_model"

)


Training: mobilenet_model
Epoch 1/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.7959 - loss: 0.4520 - val_accuracy: 0.9398 - val_loss: 0.1582
Epoch 2/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.8980 - loss: 0.2447 - val_accuracy: 0.9599 - val_loss: 0.1158
Epoch 3/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.9345 - loss: 0.1752 - val_accuracy: 0.9699 - val_loss: 0.0869
Epoch 4/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.9286 - loss: 0.1620 - val_accuracy: 0.9749 - val_loss: 0.0748
Epoch 5/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.9549 - loss: 0.1275 - val_accuracy: 0.9799 - val_loss: 0.0668
Epoch 6/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.9544 - loss: 0.1209 - val_accuracy: 0.9774 - val_loss: 0.0642
Epoch 7/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 55s 931ms/step - accuracy: 0.9565 - loss: 0.1094 - val_accuracy: 0.9850 - val_loss: 0.0537
Epoch 8/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 83s 950ms/step - accuracy: 0.9619 - loss: 0.0905 -

### Train EfficientNetB0 Model

In [7]:
# code

efficientnet_model = EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(224, 224, 3)

)

efficientnet_results = train_and_evaluate_model(

    efficientnet_model,

    "efficientnet_model"

)


Training: efficientnet_model
Epoch 1/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.5183 - loss: 0.7008 - val_accuracy: 0.5313 - val_loss: 0.7010
Epoch 2/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 402s 899ms/step - accuracy: 0.5381 - loss: 0.6978 - val_accuracy: 0.5313 - val_loss: 0.6924
Epoch 3/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 55s 930ms/step - accuracy: 0.4946 - loss: 0.6995 - val_accuracy: 0.5313 - val_loss: 0.6912
Epoch 4/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 54s 913ms/step - accuracy: 0.5161 - loss: 0.6946 - val_accuracy: 0.5313 - val_loss: 0.6930
Epoch 5/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 54s 916ms/step - accuracy: 0.5048 - loss: 0.6965 - val_accuracy: 0.5313 - val_loss: 0.6909
Epoch 6/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 54s 921ms/step - accuracy: 0.5209 - loss: 0.6929 - val_accuracy: 0.5313 - val_loss: 0.6913
Epoch 7/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 53s 903ms/step - accuracy: 0.5124 - loss: 0.6928 - val_accuracy: 0.5313 - val_loss: 0.6914
Epoch 8/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 56s 944ms/step - accuracy: 0.53

d:\Internship\Projects\Aerial-Object-Classification-and-Detection\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Internship\Projects\Aerial-Object-Classification-and-Detection\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Internship\Projects\Aerial-Object-Classification-and-Detection\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

### Show All Results

In [8]:
print("\nResNet50:", resnet_results)

print("\nMobileNet:", mobilenet_results)

print("\nEfficientNet:", efficientnet_results)


ResNet50: {'accuracy': 0.7107231920199502, 'precision': 0.7158647020947791, 'recall': 0.7107231920199502, 'f1': 0.7059196480493024, 'training_time': 66.82191548347473, 'size': 93.61585712432861}

MobileNet: {'accuracy': 0.9825436408977556, 'precision': 0.9825502464040807, 'recall': 0.9825436408977556, 'f1': 0.9825408064538108, 'training_time': 23.679771721363068, 'size': 14.135868072509766}

EfficientNet: {'accuracy': 0.5311720698254364, 'precision': 0.2821437677626383, 'recall': 0.5311720698254364, 'f1': 0.368533064732306, 'training_time': 15.766227972507476, 'size': 18.13664150238037}


### Save Results to CSV

In [9]:
# code 

import pandas as pd

results = {

    "Model": [

        "ResNet50",

        "MobileNet",

        "EfficientNetB0"

    ],

    "Accuracy": [

        resnet_results["accuracy"],

        mobilenet_results["accuracy"],

        efficientnet_results["accuracy"]

    ],

    "Precision": [

        resnet_results["precision"],

        mobilenet_results["precision"],

        efficientnet_results["precision"]

    ],

    "Recall": [

        resnet_results["recall"],

        mobilenet_results["recall"],

        efficientnet_results["recall"]

    ],

    "F1-score": [

        resnet_results["f1"],

        mobilenet_results["f1"],

        efficientnet_results["f1"]

    ],

    "Training Time (min)": [

        resnet_results["training_time"],

        mobilenet_results["training_time"],

        efficientnet_results["training_time"]

    ],

    "Model Size (MB)": [

        resnet_results["size"],

        mobilenet_results["size"],

        efficientnet_results["size"]

    ]

}

df = pd.DataFrame(results)

df.to_csv(

    "../results/transfer_learning_results.csv",

    index=False

)

df

,Model,Accuracy,Precision,Recall,F1-score,Training Time (min),Model Size (MB)
0,ResNet50,0.710723,0.715865,0.710723,0.705920,66.821915,93.615857
1,MobileNet,0.982544,0.982550,0.982544,0.982541,23.679772,14.135868
2,EfficientNetB0,0.531172,0.282144,0.531172,0.368533,15.766228,18.136642
